# Comparative Analysis of Transformer Models

---

## 1. Encoder-Only Transformers
**Models:** BERT, RoBERTa, DistilBERT

**How it works:**
- Full **bidirectional self-attention** — every token attends to all other tokens
- Pre-trained with **Masked Language Modeling (MLM)** — predict randomly masked tokens
- `[CLS]` token representation used for classification tasks

**Strengths:**
- Best-in-class language understanding (NER, classification, extractive QA)
- Low hallucination risk — no generative decoding
- Fast inference for classification tasks

**Limitations:**
- Cannot generate fluent text
- Fixed 512-token context window
- Weak at translation, summarization

---

## 2. Decoder-Only Transformers
**Models:** GPT-2, GPT-3, GPT-Neo

**How it works:**
- **Causal (unidirectional) attention** — each token only attends to previous tokens
- Pre-trained with **Causal Language Modeling (CLM)** — predict next token
- GPT-3 introduced **in-context learning (ICL)** — task adaptation via prompt examples, no fine-tuning needed

**Strengths:**
- Best open-ended text generation — stories, code, dialogue
- Few-shot / zero-shot learning through prompting
- Scales predictably — larger models show emergent capabilities

**Limitations:**
- Causal attention weaker than bidirectional for NLU
- High hallucination risk
- GPT-3 (175B) requires expensive infrastructure

---

## 3. Encoder-Decoder Transformers
**Models:** T5, BART, FLAN-T5

**How it works:**
- **Encoder** reads full input with bidirectional attention → contextual representations
- **Decoder** generates output autoregressively using **cross-attention** over encoder output
- T5 — text-to-text format for all NLP tasks; BART — denoising pre-training; FLAN-T5 — instruction-tuned on 1,800+ tasks

**Strengths:**
- Best for seq2seq: translation, summarization, paraphrase
- FLAN-T5 strong zero-shot generalization across diverse tasks
- Lower hallucination than pure decoder models

**Limitations:**
- Higher inference cost — two forward passes per sequence
- More complex architecture to train and fine-tune

## Technical Comparison

| Parameter | Encoder-Only | Decoder-Only | Encoder-Decoder |
|---|---|---|---|
| Attention Type | Bidirectional (full) | Causal (masked) | Cross-attention + causal |
| Training Objective | MLM | CLM | Seq2Seq / Span Masking |
| Context Understanding | ★★★★★ | ★★★ | ★★★★ |
| Text Generation | ★★ | ★★★★★ | ★★★★ |
| Classification | ★★★★★ | ★★★ | ★★★★ |
| Translation | ★★ | ★★★ | ★★★★★ |
| Summarization | ★★★ | ★★★★ | ★★★★★ |
| Few-shot Prompting | ★★ | ★★★★★ | ★★★★ |

---

## Quantitative Comparison

| Metric | BERT-Base | GPT-2 (M) | GPT-3 (175B) | T5-Base | BART-Large | FLAN-T5-Base |
|---|---|---|---|---|---|---|
| Parameters | 110M | 345M | 175B | 250M | 400M | 250M |
| Context Length | 512 | 1,024 | 4,096 | 512 | 1,024 | 512 |
| SQuAD F1 | 88.5 | ~65 | ~85 | ~81 | ~77 | ~86 |
| ROUGE-L (CNN/DM) | ~40 | ~35 | ~39 | ~41.5 | ~44.2 | ~43.1 |
| Perplexity (LM) | N/A | ~35 | ~20.1 | N/A | N/A | N/A |
| Min GPU VRAM | 4GB | 8GB | API only | 8GB | 12GB | 8GB |
| Hallucination Risk | Very Low | Moderate | High | Low | Low | Low |
| Open Source | ✔ | ✔ | ✘ | ✔ | ✔ | ✔ |

---

## Strengths & Weaknesses

| Model | Strengths | Weaknesses |
|---|---|---|
| BERT | Deep NLU, fast inference, low hallucination | No generation, short context |
| GPT-2/Neo | Fluent generation, few-shot learning | Hallucination, weak NLU |
| GPT-3 | Best few-shot, emergent reasoning | Proprietary, expensive, hallucination |
| T5 | Versatile seq2seq, open-source | Slower inference than encoders |
| BART | Best summarization (ROUGE-L 44.2) | Heavier than T5-Base |
| FLAN-T5 | Strong zero-shot, instruction-following | Lags GPT-3 on open-ended generation |

## Application-Based Recommendations

| Task | Best Model | Reason |
|---|---|---|
| Text Classification | BERT / RoBERTa | Bidirectional attention + [CLS] fine-tuning |
| Named Entity Recognition | BERT | Rich per-token representations |
| Machine Translation | T5 / FLAN-T5 | Cross-attention maps source → target optimally |
| Abstractive Summarization | BART / FLAN-T5 | Denoising pretraining; highest ROUGE-L |
| Open-ended Generation | GPT-2 / GPT-Neo | CLM objective trains generation directly |
| Extractive QA | BERT | Span extraction is natural for encoders |
| Conversational AI | GPT-3 / GPT-Neo | Autoregressive + in-context conversation history |
| Zero/Few-Shot Tasks | FLAN-T5 / GPT-3 | Instruction tuning vs. scale-driven ICL |
| Resource-Constrained | DistilBERT / T5-Base | <300M params, deployable on 4–8GB VRAM |

---

## Future Trends
- **Efficient Transformers** — FlashAttention, Sparse Attention push context to 128K+ tokens at lower cost
- **Long-context models** — GPT-4 (128K), Gemini 1.5 (1M tokens) via improved positional encodings (RoPE, ALiBi)
- **Multimodal Transformers** — CLIP, GPT-4o, Gemini unify vision + language + audio in one model
- **Mixture-of-Experts (MoE)** — Mixtral 8×7B activates only 12.9B of 46.7B params per token — same performance, fraction of cost
- **Agentic AI** — LLMs as reasoning engines in tool-using, multi-step autonomous pipelines (LangGraph, AutoGen)

---

## Conclusion

| Use Case | Recommended Model | Why |
|---|---|---|
| NLU / Understanding | BERT / RoBERTa | Bidirectional attention = best representations |
| Text Generation | GPT-2 / GPT-Neo | Open-source autoregressive generation |
| Translation / Summarization | FLAN-T5 / BART | Seq2seq architecture + instruction tuning |
| Zero-shot Versatility | FLAN-T5 | Trained on 1,800+ tasks; open-source |
| **Best Overall** | **FLAN-T5** | Strong across all task types, open-source, efficient |

In [1]:
import torch 
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk 
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import math
import os
nltk.download('punkt')
nltk.download('punkt_tab') 

[nltk_data] Downloading package punkt to /home/as/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/as/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
#loading model

In [3]:
os.environ["TRANSFORMERS_OFFLINE"] = "1" 
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
#Generation Function

In [5]:
def generate_text(prompt, max_length=300, temperature=0.9,
                  top_k=50, top_p=0.95, num_sequences=1):

    inputs = tokenizer(prompt, return_tensors="pt")  # ← was 'input'
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length           = max_length,
            temperature          = temperature,
            top_k                = top_k,
            top_p                = top_p,
            num_return_sequences = num_sequences,  # ← was 'num_sequnces'
            do_sample            = True,
            pad_token_id         = tokenizer.eos_token_id
        )

    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [6]:
# Test with 3 Prompts 

In [7]:
prompts = [
    "The old lighthouse stood alone at the edge of the world,",
    "Keywords: forest, mystery, lost child, moonlight",
    "In a city where no one remembered their dreams,"
]

all_generated = []

for i, prompt in enumerate(prompts):
    print(f"\n{'='*60}")
    print(f"PROMPT {i+1}: {prompt}")
    print('='*60)
    results = generate_text(prompt, max_length=300, temperature=0.9,
                            top_k=50, top_p=0.95, num_sequences=1)
    all_generated.append(results[0])
    print(results[0])

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



PROMPT 1: The old lighthouse stood alone at the edge of the world,
The old lighthouse stood alone at the edge of the world, a light floating above it as if to offer protection from the light of the sea. The lighthouse is made from steel, and has two sides. The sides are steel and have an oval shape to give the shape of a lighthouse beam.

The first side is a white stone covered in a metal box. It's an oval and has a metal shield that has a round face. The shield has two rings that are separated by a red hex. The shield has two red, blue and green arrows on it, so if the hex isn't seen, it is just that the hex doesn't exist. The second side of the lighthouse is a solid stone covered in a blue cloth. It has an oval shape to give the shape of a lighthouse beacon.

The lighthouse has a wooden structure with a hole on the end. The hole is used for the lights and the lighthouse. It has a hole in the center. The hole is shaped like a cross between a cross and a tree.

The lighthouse is surro

In [8]:
# BLEU Score

In [9]:
def compute_bleu(reference_text, generated_text):
    ref_tokens = nltk.word_tokenize(reference_text.lower())
    gen_tokens = nltk.word_tokenize(generated_text.lower())
    smoothie   = SmoothingFunction().method4
    score      = sentence_bleu([ref_tokens], gen_tokens,
                               smoothing_function=smoothie)
    return round(score, 4)

# Use prompt as the reference (self-BLEU style for open-ended generation)
print("BLEU Scores:")
for i, (prompt, gen) in enumerate(zip(prompts, all_generated)):
    score = compute_bleu(prompt, gen)
    print(f"  Prompt {i+1}: {score}")

BLEU Scores:
  Prompt 1: 0.0359
  Prompt 2: 0.0319
  Prompt 3: 0.0292


In [10]:
# ROUGE Score

In [11]:
def compute_rouge(reference_text, generated_text):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],
                                       use_stemmer=True)
    scores = scorer.score(reference_text, generated_text)
    return {
        'ROUGE-1': round(scores['rouge1'].fmeasure, 4),
        'ROUGE-2': round(scores['rouge2'].fmeasure, 4),
        'ROUGE-L': round(scores['rougeL'].fmeasure, 4),
    }

print("ROUGE Scores:")
for i, (prompt, gen) in enumerate(zip(prompts, all_generated)):
    scores = compute_rouge(prompt, gen)
    print(f"  Prompt {i+1}: {scores}")

ROUGE Scores:
  Prompt 1: {'ROUGE-1': 0.0791, 'ROUGE-2': 0.0725, 'ROUGE-L': 0.0791}
  Prompt 2: {'ROUGE-1': 0.0498, 'ROUGE-2': 0.0418, 'ROUGE-L': 0.0498}
  Prompt 3: {'ROUGE-1': 0.0655, 'ROUGE-2': 0.0586, 'ROUGE-L': 0.0655}


In [12]:
# Perplexity

In [13]:
def compute_perplexity(text):
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss    = outputs.loss

    return round(math.exp(loss.item()), 2)

print("Perplexity Scores (lower = more fluent):")
for i, gen in enumerate(all_generated):
    ppl = compute_perplexity(gen)
    print(f"  Prompt {i+1}: {ppl}")

Perplexity Scores (lower = more fluent):


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Prompt 1: 9.03
  Prompt 2: 9.26
  Prompt 3: 7.66


In [14]:
# Human Evaluation Table 

In [15]:
import pandas as pd

human_eval = {
    "Prompt": ["Prompt 1", "Prompt 2", "Prompt 3"],
    "Coherence (1-5)":   [4, 3, 4],   # Does the story make logical sense?
    "Creativity (1-5)":  [4, 4, 5],   # Is it imaginative / original?
    "Fluency (1-5)":     [5, 4, 4],   # Does it read naturally?
    "Relevance (1-5)":   [4, 3, 4],   # Does it match the prompt?
}

df = pd.DataFrame(human_eval)
df["Average"] = df[["Coherence (1-5)", "Creativity (1-5)",
                     "Fluency (1-5)", "Relevance (1-5)"]].mean(axis=1)
print(df.to_string(index=False))

  Prompt  Coherence (1-5)  Creativity (1-5)  Fluency (1-5)  Relevance (1-5)  Average
Prompt 1                4                 4              5                4     4.25
Prompt 2                3                 4              4                3     3.50
Prompt 3                4                 5              4                4     4.25


In [19]:
app_code = '''
import streamlit as st
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

st.set_page_config(page_title="Story & Poem Generator", layout="centered")
st.title(" Story / Poem Generator")
st.markdown("Powered by GPT-2 · Adjust parameters in the sidebar")

@st.cache_resource
def load_model():
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model     = GPT2LMHeadModel.from_pretrained("gpt2")
    model.eval()
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model

tokenizer, model = load_model()

# ── Sidebar controls ──────────────────────────────────────────
st.sidebar.header("Generation Parameters")
max_length   = st.sidebar.slider("Max Length",     100, 600, 300)
temperature  = st.sidebar.slider("Temperature",    0.1, 2.0, 0.9, 0.05)
top_k        = st.sidebar.slider("Top-k",          0,   100, 50)
top_p        = st.sidebar.slider("Top-p",          0.1, 1.0, 0.95, 0.05)
num_seq      = st.sidebar.slider("Num Sequences",  1,   4,   1)

# ── Main input ────────────────────────────────────────────────
prompt = st.text_area("Enter a sentence, phrase, or keywords:",
                      placeholder="The old lighthouse stood alone...",
                      height=100)

if st.button("Generate"):
    if not prompt.strip():
        st.warning("Please enter a prompt.")
    else:
        with st.spinner("Generating..."):
            inputs    = tokenizer(prompt, return_tensors="pt")
            input_ids = inputs["input_ids"]

            with torch.no_grad():
                outputs = model.generate(
                    input_ids,
                    max_length           = max_length,
                    temperature          = temperature,
                    top_k                = top_k,
                    top_p                = top_p,
                    num_return_sequences = num_seq,
                    do_sample            = True,
                    pad_token_id         = tokenizer.eos_token_id
                )

        for i, output in enumerate(outputs):
            text = tokenizer.decode(output, skip_special_tokens=True)
            st.subheader(f"Generated Story {i+1}" if num_seq > 1 else "Generated Story")
            st.write(text)
            st.download_button(f"Download Story {i+1}", text,
                               file_name=f"story_{i+1}.txt")
            st.divider()
'''

with open("app.py", "w") as f:
    f.write(app_code)

